<a href="https://colab.research.google.com/github/pedrosampaiom2007-gif/Analise-de-dados-Airbnb-Rio/blob/main/notebooks/analise_precos_airbnb_rio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/pedrosampaiom2007-gif/Analise-de-dados-Airbnb-Rio/blob/main/notebooks/analise_precos_airbnb_rio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quanto custa uma noite no Rio? Análise de preços do Airbnb

Análise dos anúncios do Airbnb no Rio de Janeiro com Python e pandas: limpeza dos dados, tratamento de outliers e respostas para 6 perguntas.

**Fonte:** [Inside Airbnb](https://insideairbnb.com/get-the-data/), Rio de Janeiro (CC BY 4.0). Data da extração: `AAAA-MM-DD`.

## Perguntas
1. Quais são os bairros mais caros e os mais baratos?
2. Quanto custa cada tipo de acomodação nos bairros mais procurados?
3. Superhosts cobram mais e são mais bem avaliados?
4. Quais são as comodidades mais comuns? Ter piscina deixa a diária mais cara?
5. Quantos anfitriões entraram no Airbnb a cada ano?
6. Quanto custa a diária média em dólar?

## Sumário
0. Configuração
1. Coleta dos dados
2. Entendimento inicial
3. Tratamento
4. Outliers
5. Transformações
6. Análises
7. Exportação
8. Conclusões

## 0. Configuração

In [259]:
# Importa as bibliotecas usadas no projeto.
# Por quê: pandas/numpy para tratar os dados, matplotlib/seaborn para os gráficos, requests para a API do Banco Central.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests

# Mostra todas as colunas ao exibir um DataFrame (a base tem mais de 70 colunas).
pd.set_option('display.max_columns', None)

## 1. Coleta dos dados

Arquivo `listings.csv.gz` (anúncios) do Rio de Janeiro, lido direto da URL.

In [260]:
# Link do listings.csv.gz do Rio (copiado de insideairbnb.com/get-the-data)
URL_LISTINGS = '/content/listings.csv'

In [261]:
# Colunas usadas na análise (as outras 60+ não são necessárias).
COLUNAS = [
    'id', 'name', 'host_id', 'host_since', 'host_response_rate', 'host_acceptance_rate',
    'host_is_superhost', 'host_identity_verified', 'host_listings_count',
    'neighbourhood_cleansed', 'latitude', 'longitude', 'property_type', 'room_type',
    'accommodates', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price',
    'minimum_nights', 'availability_365', 'number_of_reviews', 'first_review', 'last_review',
    'review_scores_rating', 'review_scores_location', 'review_scores_value', 'instant_bookable',
]

## 2. Entendimento inicial

Antes de mexer nos dados: quantas linhas, quais tipos, onde estão os nulos.

In [262]:
df=pd.read_csv(URL_LISTINGS,usecols=COLUNAS,sep=',')# lendo o arquivo

In [263]:
df.head() # Conhecendo a base de dados

,id,name,host_id,host_since,host_response_rate,host_acceptance_rate,host_is_superhost,host_listings_count,host_identity_verified,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,availability_365,number_of_reviews,first_review,last_review,review_scores_rating,review_scores_location,review_scores_value,instant_bookable
0,2992450,Luxury 2 bedroom apartment,4621559,NaN,NaN,NaN,f,1,t,THIRD WARD,42.65789,-73.75370,Entire rental unit,Entire home/apt,4,1 bath,2,2.0,"[""Essentials"", ""Smoke alarm"", ""Carbon monoxide...",$65.73,28.0,169,9,2014-07-01,2022-08-17,3.56,3.22,3.67,NaN
1,3820211,Restored Precinct in Center Sq. w/Parking,19648678,NaN,NaN,NaN,t,7,t,SIXTH WARD,42.65222,-73.76724,Entire rental unit,Entire home/apt,3,1 bath,1,1.0,"[""Smart lock"", ""Microwave"", ""Conditioner"", ""Ki...",$171.00,1.0,342,320,2014-08-15,2026-09-07,4.75,4.81,4.78,NaN
2,6623339,Center Sq. Loft in Converted Precinct w/ Parking,19648678,NaN,NaN,NaN,t,7,t,SIXTH WARD,42.65222,-73.76724,Entire rental unit,Entire home/apt,2,1 bath,1,1.0,"[""Smart lock"", ""Microwave"", ""Conditioner"", ""Ki...",$242.00,1.0,337,344,2015-06-07,2026-09-07,4.74,4.79,4.73,NaN
3,9501054,Spacious suite with full bath by Capital Center,29288920,NaN,NaN,NaN,f,2,t,SECOND WARD,42.64734,-73.75851,Private room in rental unit,Private room,2,1 private bath,1,2.0,"[""Smart lock"", ""Microwave"", ""Window AC unit"", ...",$79.00,1.0,250,479,2016-02-02,2026-09-04,4.59,4.77,4.68,NaN
4,10768745,Alb hospital area studio bath wifi. (Red),5691268,NaN,NaN,NaN,t,2,t,FOURTEENTH WARD,42.66078,-73.80553,Private room in guest suite,Private room,1,1 private bath,1,1.0,"[""Microwave"", ""Window AC unit"", ""Hangers"", ""Ca...",$50.00,1.0,142,941,2016-02-05,2026-08-30,4.84,4.93,4.91,NaN


In [264]:
df.shape #Linhasxcolunas

(500, 29)

In [265]:
# Renomeando as colunas para o português

colunas_traduzidas = {
    'id': 'id',
    'name': 'nome',
    'host_id': 'anfitriao_id',
    'host_since': 'anfitriao_desde',
    'host_response_rate': 'taxa_resposta_anfitriao',
    'host_acceptance_rate': 'taxa_aceitacao_anfitriao',
    'host_is_superhost': 'anfitriao_superhost',
    'host_identity_verified': 'anfitriao_identidade_verificada',
    'host_listings_count': 'total_anuncios_anfitriao',
    'neighbourhood_cleansed': 'bairro',
    'latitude': 'latitude',
    'longitude': 'longitude',
    'property_type': 'tipo_propriedade',
    'room_type': 'tipo_quarto',
    'accommodates': 'acomoda',
    'bathrooms_text': 'banheiros_texto',
    'bedrooms': 'quartos',
    'beds': 'camas',
    'amenities': 'comodidades',
    'price': 'preco',
    'minimum_nights': 'minimo_noites',
    'availability_365': 'disponibilidade_365',
    'number_of_reviews': 'numero_avaliacoes',
    'first_review': 'primeira_avaliacao',
    'last_review': 'ultima_avaliacao',
    'review_scores_rating': 'nota_geral',
    'review_scores_location': 'nota_localizacao',
    'review_scores_value': 'nota_custo_beneficio',
    'instant_bookable': 'reserva_instantanea'
}

df.rename(columns=colunas_traduzidas, inplace=True)
display(df.head())

,id,nome,anfitriao_id,anfitriao_desde,taxa_resposta_anfitriao,taxa_aceitacao_anfitriao,anfitriao_superhost,total_anuncios_anfitriao,anfitriao_identidade_verificada,bairro,latitude,longitude,tipo_propriedade,tipo_quarto,acomoda,banheiros_texto,quartos,camas,comodidades,preco,minimo_noites,disponibilidade_365,numero_avaliacoes,primeira_avaliacao,ultima_avaliacao,nota_geral,nota_localizacao,nota_custo_beneficio,reserva_instantanea
0,2992450,Luxury 2 bedroom apartment,4621559,NaN,NaN,NaN,f,1,t,THIRD WARD,42.65789,-73.75370,Entire rental unit,Entire home/apt,4,1 bath,2,2.0,"[""Essentials"", ""Smoke alarm"", ""Carbon monoxide...",$65.73,28.0,169,9,2014-07-01,2022-08-17,3.56,3.22,3.67,NaN
1,3820211,Restored Precinct in Center Sq. w/Parking,19648678,NaN,NaN,NaN,t,7,t,SIXTH WARD,42.65222,-73.76724,Entire rental unit,Entire home/apt,3,1 bath,1,1.0,"[""Smart lock"", ""Microwave"", ""Conditioner"", ""Ki...",$171.00,1.0,342,320,2014-08-15,2026-09-07,4.75,4.81,4.78,NaN
2,6623339,Center Sq. Loft in Converted Precinct w/ Parking,19648678,NaN,NaN,NaN,t,7,t,SIXTH WARD,42.65222,-73.76724,Entire rental unit,Entire home/apt,2,1 bath,1,1.0,"[""Smart lock"", ""Microwave"", ""Conditioner"", ""Ki...",$242.00,1.0,337,344,2015-06-07,2026-09-07,4.74,4.79,4.73,NaN
3,9501054,Spacious suite with full bath by Capital Center,29288920,NaN,NaN,NaN,f,2,t,SECOND WARD,42.64734,-73.75851,Private room in rental unit,Private room,2,1 private bath,1,2.0,"[""Smart lock"", ""Microwave"", ""Window AC unit"", ...",$79.00,1.0,250,479,2016-02-02,2026-09-04,4.59,4.77,4.68,NaN
4,10768745,Alb hospital area studio bath wifi. (Red),5691268,NaN,NaN,NaN,t,2,t,FOURTEENTH WARD,42.66078,-73.80553,Private room in guest suite,Private room,1,1 private bath,1,1.0,"[""Microwave"", ""Window AC unit"", ""Hangers"", ""Ca...",$50.00,1.0,142,941,2016-02-05,2026-08-30,4.84,4.93,4.91,NaN


In [266]:
df.info()#Conhecendo os tipos de dados

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 29 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   id                               500 non-null    int64  
 1   nome                             500 non-null    object 
 2   anfitriao_id                     500 non-null    int64  
 3   anfitriao_desde                  0 non-null      float64
 4   taxa_resposta_anfitriao          0 non-null      float64
 5   taxa_aceitacao_anfitriao         0 non-null      float64
 6   anfitriao_superhost              500 non-null    object 
 7   total_anuncios_anfitriao         500 non-null    int64  
 8   anfitriao_identidade_verificada  500 non-null    object 
 9   bairro                           500 non-null    object 
 10  latitude                         500 non-null    float64
 11  longitude                        500 non-null    float64
 12  tipo_propriedade      

In [267]:
df.isna().sum().sum()# identificando quantos valores nulos há

np.int64(2296)

In [268]:
# Salvando em uma variável os dados nulos
df_nulos=df.isna().stack()

In [269]:
#Aplicando a máscara ou seja filtrando o resultado do outro filtro
df_nulos = df_nulos[df_nulos]


In [270]:
# listando a localização dos dados nulos pelo indice
lista_nul = df_nulos.index.to_list()


## 3. Tratamento

Em cada etapa que remove linhas, mostrar quantas linhas existiam antes e depois.

### 3.1 Duplicadas

In [271]:
df.duplicated().sum() # Nossa base de dados não possui duplicadas, portanto não há o que remover

np.int64(0)

### 3.2 Preço
O preço vem como texto: `"$1,234.00"`. É preciso remover `$` e `,` e converter para float.

In [272]:
df['preco']

,preco
0,$65.73
1,$171.00
2,$242.00
3,$79.00
4,$50.00
...,...
495,$197.00
496,$185.00
497,$139.20
498,$356.00


In [273]:
df['preco']=df['preco'].str.replace('$','',regex=False).str.replace(',','',regex=False)

In [274]:
df['preco']=df['preco'].astype(float)

### 3.3 Percentuais
`"95%"` vira `0.95`.

In [275]:
df_percent=(df[['taxa_resposta_anfitriao', 'taxa_aceitacao_anfitriao']]=='%')

In [276]:
df_percent.sum().sum() # não existe valores percentuais nas colunas de taxa

np.int64(0)

### 3.4 Colunas t/f

In [277]:
mapeamento_de_cols={

                    't':True,
                    'f':False}

In [278]:
# Aqui foi feito uma substituição onde os valores eram apenas t e f, viraram True e False para facilitar o tratamento e leitura dos dados de 'anfitriao_superhost'
df['anfitriao_superhost']=df['anfitriao_superhost'].replace(mapeamento_de_cols)

/tmp/ipykernel_774/220183403.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['anfitriao_superhost']=df['anfitriao_superhost'].replace(mapeamento_de_cols)


In [279]:
df['anfitriao_superhost'] # visualização da alteração dos dados, nota-se que se tornou do tipo bool após o tratamento da coluna

,anfitriao_superhost
0,False
1,True
2,True
3,False
4,True
...,...
495,False
496,True
497,False
498,False


### 3.5 Datas

In [280]:
datas_null=df[['primeira_avaliacao','ultima_avaliacao']]


In [281]:
df_aval=df[(df['primeira_avaliacao'].isna()) & (df['ultima_avaliacao'].isna())]

In [282]:
df_sem_av_na = df.drop(index=df_aval.index).copy()
df_sem_av_na['primeira_avaliacao'] = pd.to_datetime(df_sem_av_na['primeira_avaliacao'])
df_sem_av_na['ultima_avaliacao'] = pd.to_datetime(df_sem_av_na['ultima_avaliacao'])


In [283]:
df_sem_av_na[['primeira_avaliacao','ultima_avaliacao']].info()  # agora devem ser datetime64

<class 'pandas.core.frame.DataFrame'>
Index: 449 entries, 0 to 497
Data columns (total 2 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   primeira_avaliacao  449 non-null    datetime64[ns]
 1   ultima_avaliacao    449 non-null    datetime64[ns]
dtypes: datetime64[ns](2)
memory usage: 10.5 KB


### 3.6 Banheiros
`"1.5 baths"` vira `1.5`.

In [284]:
# Substituindo 'Half-bath' por '0.5' para que possa ser convertido em número
df['banheiros_texto'] = df['banheiros_texto'].str.replace('(?i)half-bath', '0.5', regex=True)

# Extraindo apenas os números (ponto flutuante ou inteiros) ignorando textos e espaços
df['banheiros_texto'] = df['banheiros_texto'].str.extract(r'(\d+\.?\d*)')[0]

# Convertendo a coluna para tipo numérico correto (float) para aceitar valores decimais (ex: 1.5)
df['banheiros_texto'] = df['banheiros_texto'].astype(float)

# Visualizando o resultado
df['banheiros_texto'].head()

,banheiros_texto
0,1.0
1,1.0
2,1.0
3,1.0
4,1.0


### 3.7 Nulos nas avaliações
Anúncios sem nenhuma avaliação têm `review_scores_*` nulos. *(decisão e justificativa)*

In [289]:
df

In [293]:
df_sem=df.dropna(subset=['nota_geral',	'nota_localizacao',	'nota_custo_beneficio']) .copy()
# Foi decidio remover as colunas  com scores= Nan pois  sem avaliacoes nao há resultados, é desnecessário manter esse tipo de dado na nossa base

## 4. Outliers de preço

**Decisão sobre os outliers:** *(explicar aqui o que foi feito e por quê)*

## 5. Transformações

### 5.1 Comodidades
`'["Wifi", "TV", "Kitchen"]'` é uma string, não uma lista.

### 5.2 Ano de entrada do anfitrião
Nova coluna com o ano de `host_since`.

### 5.3 Conferência final do tratamento
Tipos e nulos depois de todas as etapas.

## 6. Análises

### P1. Os 10 bairros mais caros e os 10 mais baratos
Preço médio por noite (já sem os outliers), considerando só bairros com pelo menos 30 anúncios.

### P2. Preço por tipo de acomodação nos bairros com mais anúncios

### P3. Superhosts cobram mais e têm avaliações melhores?

### P4. Comodidades mais comuns e o efeito da piscina no preço

### P5. Quantos anfitriões entraram no Airbnb a cada ano?

### P6. Diária média em dólar
Cotação do dólar pela API do Banco Central (série 1, dólar comercial).

In [285]:
URL_DOLAR = 'https://api.bcb.gov.br/dados/serie/bcdata.sgs.1/dados/ultimos/1?formato=json'

## 7. Exportação

Salva a base tratada em CSV e em um banco SQLite.

## 8. Conclusões

| Pergunta | Resposta |
| --- | --- |
| P1 | |
| P2 | |
| P3 | |
| P4 | |
| P5 | |
| P6 | |

**Limitações:** *(ex.: preço do anúncio não é o preço pago; dados de uma única extração...)*

**Próximos passos:** *(ex.: modelo de previsão de preço, comparar com outras cidades...)*